In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import time

In [0]:
def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception as e:
        if 'java.io.FileNotFoundException' in str(e):
            return False
        else:
            raise e

In [0]:
class CourseDataset:
    def __init__(self,uri, data_catalog, db_name, location = None, checkpoint = None):
        self.uri = uri
        self.dataset_path = location
        self.checkpoint_path = checkpoint
        self.catalog_name = data_catalog
        self.db_name = db_name
    
    def download_dataset(self):
        source = self.uri
        target = self.dataset_path

        if self.catalog_name == "hive_metastore":
            try:
                spark.config.set("fs.s3a.endpoint", "s3.eu-west-3.amazonaws.com")
                spark.config.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.AnonymousAWSCreadentialsProvider")
            except:
                pass 

        files = dbutils.fs.ls(source)

        for f in files:
            source_path = f"{source}/{f.name}"
            target_path = f"{target}/{f.name}"
            if not path_exists(target_path):
                print(f"Copying {f.name} ...")
                dbutils.fs.cp(source_path, target_path, True)

    def __create_volume(self):
        dataset_volume_name = "dataset"
        checkpoints_volume_name = "checkpoints"
        volume_root = f"/Volumes/{self.catalog_name}/{self.db_name}"
        self.dataset_path = f"{volume_root}/{dataset_volume_name}"
        self.checkpoint_path = f"{volume_root}/{checkpoints_volume_name}"

        spark.sql(f"create volume if not exists {dataset_volume_name}")
        spark.sql(f"create volume if not exists {checkpoints_volume_name}") 

    def create_database(self):
        spark.sql(f"use catalog {self.catalog_name}")
        spark.sql(f"create schema if not exists {self.db_name}")
        spark.sql(f"use schema {self.db_name}")

        print(f"Data catalog: {self.catalog_name}")
        print(f"Schema: {self.db_name}" )

        if self.catalog_name != "hive_metastore": 
            self.__create_volume()  
    
    def __get_index(self,dir):
        try:
            files = dbutils.fs.ls(dir)
            file = max(f.name for f in files if f.name.endswith(".json"))
            index = int(file.rsplit('.',maxsplit = 1)[0])
        except:
            index = 0
        return index + 1
    
    def __load_json_files(self,current_index, streaming_dir,raw_dir):
        latest_file = f"{str(current_index).zfill(2)}.json"
        source = f"{streaming_dir}/{latest_file}"
        target = f"{raw_dir}/{latest_file}"
        prefix = steaming_dir.split("/")[-1]
        if path_exists(source):
            print(f"loading {prefix} - {latest_file} file to the bookstore dataset")
            dbutils.fs.cp(source,target)


    def __load_data(self, max, streaming_dir, raw_dir, all = False):
        index = self.__get_indec(raw_dir)
        if index > max:
            print("no more data to load\n")
        
        elif all == True:
            while index <=max:
                self.__load_jason_file(index,streaming_dir, raw_dir)
                index += 1
        else:
            self.__load_jason_file(index,streaming_dir, raw_dir)
            index += 1
    
    def load_new_data(self, num_files = 1):
        streaming_dir = f"{self.dataset_path}/kafka-streaming"
        raw_dir = f"{self.dataset_path}/kafka-raw"
        for i in range(num_files):
            self.__load_data(10,streaming_dir, raw_dir)
    
    def clean_up(self):
        if self.catalog_name == "hive_metastore":
            print("Removing checkpoint...")
            dbutils.fs.rm(self.checkpoint_path, True)
            print("Dropping databases ...")
            spark.sql(f"drop schema if exists {self.db_name} cascade")
            print("Removing Dataset ...")
            dbutils.fs.rm(self.dataset_path, True)
        else:
            print("Dropping Databases, Datasets, and Checkpoints ...")
            spark.sql(f"drop schema if exists {self.db_name} cascade")
        print("Done")

        




In [0]:
data_source_uri = "s3://dalhussein-courses/DE-Pro/datasets/bookstore/v1/"
db_name = "bookstore_eng_pro"

catalogs = spark.sql("SHOW CATALOGS").collect()
hive_exists = any(row.catalog == 'hive_metastore' for row in catalogs)
if hive_exists:
    data_catalog = 'hive_metastore'
    bookstore_dataset = 'dbfs:/mnt/demo-datasets/DE-Pro/bookstore'
    bookstore_checkpoint = "dbfs:/mnt/demo_pro/checkpoints"

    bookstore = CourseDataset(data_source_uri, data_catalog, db_name, bookstore_dataset, bookstore_checkpoint)
else:
    data_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
    bookstore = CourseDataset(data_source_uri, data_catalog, db_name)

bookstore.create_database()
bookstore.download_dataset()

In [0]:
#display(dbutils.fs.ls("s3://dalhussein-courses/DE-Pro/datasets/bookstore/v1/"))